# Notebook 05c — Restaurant Song Recommender (simplified, plain-language version)

This is a **readable rewrite** of `05b_recommender.ipynb` — same math, same results,
written for someone who doesn't code. Every technical step in 05b is still here, just:

- duplicated code merged into one reusable function
- long printed loops replaced with tables
- jargon moved into plain-English explanations

For the *why* behind the approach (and its limitations), see
`notes/weight_matrix_methodology.md`. This notebook is the *how*, condensed.


In [1]:
import re
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from collections import Counter

PROCESSED = r"C:\Users\ryanm\Documents\coding_projects\restaurant_recommendation\data\processed"

songs = pd.read_csv(os.path.join(PROCESSED, "song_pca.csv")).drop_duplicates(subset=["name", "artists"]).reset_index(drop=True)
restaurants = pd.read_csv(os.path.join(PROCESSED, "restaurant_pca.csv"))
yelp_raw = pd.read_csv(os.path.join(PROCESSED, "yelp_ml_ready.csv"))

pca_spotify = joblib.load(os.path.join(PROCESSED, "spotify_pca.pkl"))
scaler_spotify = joblib.load(os.path.join(PROCESSED, "spotify_scaler.pkl"))
pca_yelp = joblib.load(os.path.join(PROCESSED, "yelp_pca.pkl"))

SPOTIFY_PC = [c for c in songs.columns if c.startswith("pc")]
YELP_PC = [c for c in restaurants.columns if c.startswith("pc")]
song_vecs = songs[SPOTIFY_PC].values
restaurant_vecs = restaurants[YELP_PC].values

print(f"{len(songs):,} songs  x  {len(SPOTIFY_PC)} music traits")
print(f"{len(restaurants):,} restaurants  x  {len(YELP_PC)} vibe traits")


543,319 songs  x  6 music traits
34,808 restaurants  x  13 vibe traits


## Step 1 — What do the numbers actually mean?

PCA compresses many raw columns (danceability, noise level, ambience...) into a handful
of "traits." Each trait is really just a blend of the original columns — this step
prints, in plain terms, what each trait is mostly made of.


In [2]:
AUDIO_FEATURES = [
    "danceability", "energy", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence", "loudness", "tempo",
][:pca_spotify.n_features_in_]

YELP_FEATURES = [
    "Ambience.romantic", "Ambience.divey", "Ambience.classy",
    "Ambience.hipster", "Ambience.trendy", "Ambience.upscale", "Ambience.casual",
    "HasTV", "HappyHour", "RestaurantsGoodForGroups",
    "GoodForMeal.breakfast", "GoodForMeal.brunch",
    "GoodForMeal.latenight", "GoodForMeal.dinner",
    "RestaurantsTableService", "NoiseLevel", "stars",
][:pca_yelp.n_features_in_]

def top_traits(components, pc_names, feature_names, n=2):
    """For each PC, name the 1-2 raw features that drive it most."""
    loadings = pd.DataFrame(components, index=pc_names, columns=feature_names)
    summary = {}
    for pc in pc_names:
        row = loadings.loc[pc].reindex(loadings.loc[pc].abs().sort_values(ascending=False).index)
        summary[pc] = ", ".join(f"{feat} ({val:+.2f})" for feat, val in row.head(n).items())
    return pd.Series(summary, name="dominated by")

YELP_PC_COVERED = [f"pc{i}" for i in range(1, 9)]  # the 8 traits with a clean human label
display(top_traits(pca_yelp.components_[:len(YELP_PC_COVERED)], YELP_PC_COVERED, YELP_FEATURES).to_frame())
display(top_traits(pca_spotify.components_, SPOTIFY_PC, AUDIO_FEATURES).to_frame())

# only pc1-pc8 have a clean plain-English meaning (above) -- everything downstream
# matches restaurants to music using just these 8, not all 13 Yelp traits
restaurant_vecs = restaurants[YELP_PC_COVERED].values


,dominated by
pc1,"GoodForMeal.dinner (+0.42), RestaurantsTableSe..."
pc2,"GoodForMeal.breakfast (+0.63), GoodForMeal.bru..."
pc3,"Ambience.romantic (+0.40), Ambience.casual (-0..."
pc4,"NoiseLevel (+0.50), stars (-0.44)"
pc5,"Ambience.hipster (+0.61), Ambience.trendy (+0.41)"
pc6,"Ambience.divey (+0.77), GoodForMeal.latenight ..."
pc7,"GoodForMeal.latenight (+0.63), HasTV (-0.38)"
pc8,"RestaurantsGoodForGroups (+0.64), Ambience.div..."


,dominated by
pc1,"energy (+0.48), loudness (+0.48)"
pc2,"danceability (+0.55), valence (+0.42)"
pc3,"liveness (+0.71), speechiness (+0.58)"
pc4,"tempo (+0.90), speechiness (+0.31)"
pc5,"speechiness (+0.64), liveness (-0.50)"
pc6,"instrumentalness (+0.78), valence (+0.42)"


## Step 2 — Name the 8 restaurant "vibes" we're targeting

Only 8 of the 13 restaurant traits have an obvious plain-English name. Those 8 become
our **vibe archetypes** — the categories every restaurant ultimately gets matched
against.


In [3]:
VIBE_LABEL = {
    "pc1": "sit-down dinner",
    "pc2": "brunch / breakfast",
    "pc3": "fine dining",
    "pc4": "loud / chaotic, lower-rated",
    "pc5": "hipster / trendy",
    "pc6": "dive bar",
    "pc7": "late-night",
    "pc8": "good for groups",
}
MUSIC_LABEL = {
    "pc1": "energetic + loud + danceable",
    "pc2": "happy + danceable (acoustic-leaning)",
    "pc3": "live performance + speechy",
    "pc4": "fast tempo",
    "pc5": "spoken word / pure instrumental",
    "pc6": "live instrumental / jazz-like",
}
pd.Series(VIBE_LABEL, name="vibe").to_frame()


,vibe
pc1,sit-down dinner
pc2,brunch / breakfast
pc3,fine dining
pc4,"loud / chaotic, lower-rated"
pc5,hipster / trendy
pc6,dive bar
pc7,late-night
pc8,good for groups


## Step 3 — Give each vibe a real-music example

Instead of guessing "fine dining music sounds like X," we point to real, recognizable
artists for each vibe (Miles Davis for fine dining, Daft Punk for good-for-groups...)
and average their *actual* songs' music traits. That average becomes the target sound
for that vibe. If a vibe doesn't have enough matching songs in our catalog, it falls
back to a hand-typed guess instead — flagged below wherever that happens.


In [4]:
ANCHOR_ARTISTS = {
    "pc1": ["Bruno Mars", "John Legend", "Michael Buble", "Jason Mraz", "Andra Day",
            "Alicia Keys", "Adele", "Ben Rector", "Colbie Caillat", "Gavin DeGraw"],
    "pc2": ["Leon Bridges", "Norah Jones", "Jack Johnson", "John Mayer", "The Lumineers",
            "Maggie Rogers", "Vance Joy", "Kacey Musgraves", "Fleet Foxes", "Hozier"],
    "pc3": ["Miles Davis", "John Coltrane", "Frank Sinatra", "Ella Fitzgerald", "Bill Evans",
            "Sade", "Diana Krall", "Nat King Cole", "Stan Getz", "Nina Simone"],
    "pc4": ["AC/DC", "Guns N' Roses", "Red Hot Chili Peppers", "Kings of Leon",
            "The White Stripes", "Foo Fighters", "Weezer", "blink-182", "Fall Out Boy", "Panic! At The Disco"],
    "pc5": ["Tame Impala", "Khruangbin", "Thundercat", "Vampire Weekend", "Mac DeMarco",
            "Arctic Monkeys", "Phoebe Bridgers", "LCD Soundsystem", "The Strokes", "Blood Orange"],
    "pc6": ["Tom Waits", "The Black Keys", "Alabama Shakes", "Gary Clark Jr.", "ZZ Top",
            "Jack White", "The Rolling Stones", "Bob Seger", "Creedence Clearwater Revival", "Whiskey Myers"],
    "pc7": ["FKA twigs", "Billie Eilish", "Frank Ocean", "James Blake", "Rhye",
            "Bon Iver", "Daniel Caesar", "SZA", "Jorja Smith", "Lianne La Havas"],
    "pc8": ["Daft Punk", "Disclosure", "Calvin Harris", "The Weeknd", "Dua Lipa",
            "Fred again..", "RUFUS DU SOL", "KAYTRANADA", "Jamie xx", "Peggy Gou"],
}
MIN_ANCHOR_SONGS = 5

# fallback guesses (raw audio-feature units), only used when a vibe has too few real anchor songs
SKEWED = ["instrumentalness", "acousticness", "speechiness", "liveness"]
_mean_raw = scaler_spotify.mean_.copy()
for f in SKEWED:
    _mean_raw[AUDIO_FEATURES.index(f)] = np.expm1(_mean_raw[AUDIO_FEATURES.index(f)])
NEUTRAL = dict(zip(AUDIO_FEATURES, _mean_raw))

def guess_target(**deviations):
    intent = {**NEUTRAL, **deviations}
    row = pd.DataFrame([[intent[f] for f in AUDIO_FEATURES]], columns=AUDIO_FEATURES)
    row[SKEWED] = row[SKEWED].apply(np.log1p)
    return pca_spotify.transform(scaler_spotify.transform(row))[0]

FALLBACK_GUESS = {
    "pc1": guess_target(acousticness=0.40, valence=0.50, tempo=110.0),
    "pc4": guess_target(energy=0.80, speechiness=0.12, acousticness=0.15, instrumentalness=0.05,
                         liveness=0.35, loudness=-5.0, tempo=130.0),
    "pc6": guess_target(energy=0.70, acousticness=0.25, instrumentalness=0.15, liveness=0.45, loudness=-7.0),
    "pc7": guess_target(danceability=0.40, energy=0.35, acousticness=0.40, instrumentalness=0.40,
                         valence=0.30, loudness=-12.0, tempo=95.0),
}

rows, sources = [], {}
for pc, artists in ANCHOR_ARTISTS.items():
    mask = pd.Series(False, index=songs.index)
    for artist in artists:
        mask |= songs["artists"].str.contains(re.escape(artist), case=False, na=False)
    n_matched = int(mask.sum())
    if n_matched >= MIN_ANCHOR_SONGS:
        rows.append(song_vecs[mask.values].mean(axis=0))
        sources[pc] = f"{n_matched} real songs"
    else:
        rows.append(FALLBACK_GUESS[pc])
        sources[pc] = "hand-typed guess (not enough real songs matched)"

Y_train = np.array(rows)
pd.Series(sources, name="target source").to_frame()


,target source
pc1,256 real songs
pc2,183 real songs
pc3,349 real songs
pc4,279 real songs
pc5,261 real songs
pc6,213 real songs
pc7,228 real songs
pc8,355 real songs


## Step 4 — Build the bridge: restaurant vibe → music vibe

`W` is the matrix that translates an 8-number restaurant-vibe score into a 6-number
music-vibe score. With one clean example per vibe (step 3) and one input row per vibe
(a simple "this row *is* vibe #4" marker), fitting a straight line through them
*is* `W` — `sklearn`'s linear regression just assembles it for us.


In [5]:
X_train = np.eye(len(YELP_PC_COVERED))  # row i = "this example is 100% vibe i"
W = LinearRegression(fit_intercept=False).fit(X_train, Y_train).coef_  # shape: (6 music traits, 8 vibes)

pd.DataFrame(W, index=SPOTIFY_PC, columns=YELP_PC_COVERED).round(2)


,pc1,pc2,pc3,pc4,pc5,pc6,pc7,pc8
pc1,0.26,-0.27,-0.49,1.43,0.39,0.89,-0.41,1.46
pc2,0.32,0.20,0.20,-0.60,0.12,0.20,0.44,0.29
pc3,0.02,-0.17,0.54,0.19,-0.13,0.04,0.23,-0.36
pc4,-0.19,-0.10,0.12,-0.19,-0.06,0.01,-0.09,-0.29
pc5,-0.20,-0.46,-0.40,-0.42,-0.33,-0.24,-0.01,0.19
pc6,-0.56,-0.58,0.24,-0.25,-0.06,-0.19,-0.54,0.13


## Step 5 — One function: restaurant in, songs out

Every later step in 05b repeated the same four moves (scale the restaurant's vibe →
translate it through `W` → find the closest real songs → optionally downweight
overused songs). Here it's written once.


In [6]:
song_scaler = StandardScaler().fit(song_vecs)
song_finder = NearestNeighbors(n_neighbors=10, metric="euclidean").fit(song_scaler.transform(song_vecs))

def recommend_songs(yelp_vibe_vector, k=3, penalty=None):
    """Translate one restaurant's vibe vector into its top-k closest real songs."""
    normalized = yelp_vibe_vector / (np.linalg.norm(yelp_vibe_vector) + 1e-9)
    target_music_vibe = song_scaler.transform((W @ normalized).reshape(1, -1))
    distances, idx = song_finder.kneighbors(target_music_vibe, n_neighbors=len(songs) if penalty is not None else k)
    idx, distances = idx[0], distances[0]
    if penalty is not None:
        order = np.argsort(distances * penalty[idx])[:k]
        idx, distances = idx[order], (distances * penalty[idx])[order]
    return pd.DataFrame({
        "song": songs.loc[idx, "name"].values,
        "artist": songs.loc[idx, "artists"].values,
        "distance": distances.round(3),
    })


## Step 6 — Try it on the most extreme real example of each vibe

For each of the 8 vibes, find the single real restaurant that scores highest on it,
and see what it gets recommended.


In [7]:
boss_idx = {pc: restaurants[pc].idxmax() for pc in YELP_PC_COVERED}

results = []
for pc, idx in boss_idx.items():
    recs = recommend_songs(restaurant_vecs[idx])
    recs.insert(0, "restaurant", restaurants.loc[idx, "name"])
    recs.insert(1, "vibe", VIBE_LABEL[pc])
    results.append(recs)

pd.concat(results, ignore_index=True)


,restaurant,vibe,song,artist,distance
0,The BAO,sit-down dinner,Dead Bitch,['Evil Pimp'],0.186
1,The BAO,sit-down dinner,D-Milk (Nine Ounce Nails Mix),['Andy Colvin'],0.193
2,The BAO,sit-down dinner,D-Milk (Live @ Kcmu 1989),"['Andy Colvin', 'Ed Hall']",0.193
3,Living Room Coffee & Kitchen,brunch / breakfast,Cry Cry Cry,['Ty Segall'],0.173
4,Living Room Coffee & Kitchen,brunch / breakfast,Feels Like Heaven,['Ulrich Ellison and Tribe'],0.187
5,Living Room Coffee & Kitchen,brunch / breakfast,Dimma i dag,"['Jan Johansson', 'Georg Riedel']",0.207
6,1799 Kitchen and Cocktails,fine dining,Sunshine Moonwalk,['Silent Sabotuers'],0.243
7,1799 Kitchen and Cocktails,fine dining,Ain't no More Flowers,['Deadbeat'],0.243
8,1799 Kitchen and Cocktails,fine dining,BKLYNLDN,['Shura'],0.351
9,Effervescence,"loud / chaotic, lower-rated",Unchain My Heart,"['Big Daddy Wilson', 'Vanessa Collier', 'Si Cr...",0.199


## Step 7 — Fix the "broken jukebox" problem

Without a fix, a handful of songs end up recommended to *way* more restaurants than
makes sense — like a jukebox that keeps playing the same 5 songs no matter who walks
in. This step counts, across every restaurant, how often each song shows up in the
top 10, and flags the ones that show up for more than 5% of all restaurants
("hub songs"). Those songs then get pushed down the list — proportionally to how
overused they are — instead of removed outright.


In [8]:
all_normalized = restaurant_vecs / (np.linalg.norm(restaurant_vecs, axis=1, keepdims=True) + 1e-9)
all_targets = song_scaler.transform(all_normalized @ W.T)
_, top10 = song_finder.kneighbors(all_targets)

in_degree = Counter(top10.flatten())
hub_threshold = 0.05 * len(restaurants)

penalty = np.ones(len(songs))
in_degree_arr = np.zeros(len(songs))
for song_idx, count in in_degree.items():
    in_degree_arr[song_idx] = count
    if count > hub_threshold:
        penalty[song_idx] = np.sqrt(count / hub_threshold)

hub_songs = songs.loc[in_degree_arr > hub_threshold, ["name", "artists"]].copy()
hub_songs["shows up for this many restaurants"] = in_degree_arr[in_degree_arr > hub_threshold].astype(int)
hub_songs.sort_values("shows up for this many restaurants", ascending=False).head(10)


,name,artists,shows up for this many restaurants
454997,I'll remember you,['Duskus'],4754
81649,Run It,"['Project Paradis', 'Mr. Carmack', 'Promnite']",4006
94733,Fox's Sunday Best,"['Naomi Randall', 'Tom Gaskell']",3756
154220,VIVID DREAMS,"['KAYTRANADA', 'River Tiber']",3199
330167,We're Remembering Right Now,"['Koss', 'Henriksson', 'Mullaert']",3146
484353,Temporal,['Buscabulla'],3050
25114,Intro,['M.M.M.F.D.'],2978
506612,New Beginnings,['Hydelic'],2939
350792,BRANDENBURG - Stimming Remix,"['Apparat', 'Stimming']",2818
453802,Swirling Skies,['Peter Pearson'],2690


## Step 8 — Same 8 vibes, before vs. after the fix

The 8 "most extreme" restaurants from Step 6 are outliers by nature, so the fix rarely
changes their results — shown here for completeness. The real effect shows up on
ordinary restaurants (a random sample of 15, below).


In [9]:
before_after = []
for pc, idx in boss_idx.items():
    before = recommend_songs(restaurant_vecs[idx])
    after = recommend_songs(restaurant_vecs[idx], penalty=penalty)
    before.insert(0, "when", "before fix")
    after.insert(0, "when", "after fix")
    combo = pd.concat([before, after], ignore_index=True)
    combo.insert(0, "restaurant", restaurants.loc[idx, "name"])
    before_after.append(combo)

pd.concat(before_after, ignore_index=True)


,restaurant,when,song,artist,distance
0,The BAO,before fix,Dead Bitch,['Evil Pimp'],0.186
1,The BAO,before fix,D-Milk (Nine Ounce Nails Mix),['Andy Colvin'],0.193
2,The BAO,before fix,D-Milk (Live @ Kcmu 1989),"['Andy Colvin', 'Ed Hall']",0.193
3,The BAO,after fix,Dead Bitch,['Evil Pimp'],0.186
4,The BAO,after fix,D-Milk (Nine Ounce Nails Mix),['Andy Colvin'],0.193
5,The BAO,after fix,D-Milk (Live @ Kcmu 1989),"['Andy Colvin', 'Ed Hall']",0.193
6,Living Room Coffee & Kitchen,before fix,Cry Cry Cry,['Ty Segall'],0.173
7,Living Room Coffee & Kitchen,before fix,Feels Like Heaven,['Ulrich Ellison and Tribe'],0.187
8,Living Room Coffee & Kitchen,before fix,Dimma i dag,"['Jan Johansson', 'Georg Riedel']",0.207
9,Living Room Coffee & Kitchen,after fix,Cry Cry Cry,['Ty Segall'],0.173


In [10]:
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(restaurants), size=15, replace=False)

changed_count = 0
for idx in sample_idx:
    before = recommend_songs(restaurant_vecs[idx])["song"].tolist()
    after = recommend_songs(restaurant_vecs[idx], penalty=penalty)["song"].tolist()
    changed_count += before != after

print(f"{changed_count} / {len(sample_idx)} ordinary restaurants had their top-3 change once overused songs were downweighted.")


7 / 15 ordinary restaurants had their top-3 change once overused songs were downweighted.


## Step 9 — Try a real, fully-described restaurant

Picking a restaurant completely at random risks landing on one with almost no vibe
information filled in — the recommendation would just reflect noise. This filters to
restaurants with at least 4 vibe attributes set, then picks one at random and shows
its recommendations end-to-end.


In [11]:
BINARY_VIBE_COLS = [c for c in YELP_FEATURES if c not in ("NoiseLevel", "stars")]
richness = yelp_raw[BINARY_VIBE_COLS].sum(axis=1)
candidates = restaurants[restaurants["business_id"].isin(yelp_raw.loc[richness >= 4, "business_id"])]

pick = candidates.sample(1, random_state=np.random.default_rng().integers(0, 1_000_000)).index[0]
name = restaurants.loc[pick, "name"]

print(f"Restaurant: {name}")
recommend_songs(restaurant_vecs[pick], k=5, penalty=penalty)


Restaurant: Knockouts Bar and Hookah Lounge


,song,artist,distance
0,Ghasi Ram Blues (Desert Dwellers Remix),"['Kaya Project', 'Desert Dwellers']",0.292
1,If You Were God,"['Akira The Don', 'Alan Watts']",0.295
2,0.5%,['Dave Chose'],0.306
3,Jumpsuit,['Twenty One Pilots'],0.316
4,Make Me An Offer I Cannot Refuse,['Sufjan Stevens'],0.356


## Step 10 — Why did it recommend that? (explainability)

Two weight matrices are chained together in this whole pipeline: how raw restaurant
features roll up into the 13 vibe traits (straight from PCA, not hand-authored), and
`W` (hand-authored/anchor-grounded, Steps 3-4). Multiplying them collapses the chain
into one table: how much each *individual, raw* restaurant attribute
(`Ambience.classy`, `GoodForMeal.brunch`, ...) pushes toward each music trait — no PCA
math required to read it.


In [12]:
raw_to_music = W @ pca_yelp.components_[:len(YELP_PC_COVERED)]  # (6 music traits, 17 raw restaurant attributes)
raw_to_music_df = pd.DataFrame(raw_to_music, index=SPOTIFY_PC, columns=YELP_FEATURES)

scaler_yelp = joblib.load(os.path.join(PROCESSED, "yelp_scaler.pkl"))
raw_row = yelp_raw.loc[yelp_raw["business_id"] == restaurants.loc[pick, "business_id"], YELP_FEATURES]
raw_row = raw_row.fillna(yelp_raw[YELP_FEATURES].median())
scaled_row = scaler_yelp.transform(raw_row)[0]

dominant_trait = SPOTIFY_PC[np.argmax(np.abs(song_scaler.transform((W @ (restaurant_vecs[pick] / (np.linalg.norm(restaurant_vecs[pick]) + 1e-9))).reshape(1, -1))[0]))]

contribution = pd.Series(raw_to_music_df.loc[dominant_trait].values * scaled_row, index=YELP_FEATURES)
contribution = contribution.reindex(contribution.abs().sort_values(ascending=False).index)

print(f'For "{name}", the dominant music trait was: {dominant_trait} ({MUSIC_LABEL[dominant_trait]})')
print("Which raw restaurant attributes drove that, most important first:")
contribution.round(3).to_frame("contribution")


For "Knockouts Bar and Hookah Lounge", the dominant music trait was: pc2 (happy + danceable (acoustic-leaning))


Which raw restaurant attributes drove that, most important first:


,contribution
NoiseLevel,-1.341
HappyHour,-0.406
Ambience.casual,0.355
GoodForMeal.dinner,-0.223
HasTV,-0.200
RestaurantsGoodForGroups,0.069
GoodForMeal.latenight,-0.036
Ambience.romantic,-0.028
Ambience.hipster,-0.024
stars,-0.014
